###Notebook-0: Data-acquisition of "Die Bombe"
For the acquisition of the relevant data - in this case, each page of all the issues of the viennese periodical "Die Bombe" - the provided IIIF-manifests of the austrian national library will be used. The CSV-file provides the manifests to all issues of the historical satirical periodical "Die Bombe", which is published via the [ANNO catalogue](https://anno.onb.ac.at/) of the national library.

This Notebook will download each Canvas/page of the periodical in jpg-format and create a metadata-sheet with the following information:

- Page-ID: unique identifier of each single page (Anno-ID+0+number of page)
- Anno-ID: ID assigned per issue by the ONB
- Anno-URL: linking to the issue in the ANNO-catalogue
- Periodical-title: IIIF-manifest: metadata --> label/en/"title" --> value/en
- Date: IIIF-manifest: metadata --> label/en/"year/date" --> value/en
- Publishing-Place: IIIF-manifest: metadata --> label/en/"place" --> value/en
- Language: IIIF-manifest: metadata --> label/en/"languages" --> value/en
- Mediatype: IIIF-manifest: metadata --> label/en/"mediatype" --> value/en
- Keywords: IIIF-manifest: metadata --> label/en/"keywords" --> value/en
- Attribution: IIIF-manifest: requiredStatement --> label/en/"Attribution" --> value/en
- Provider: IIIF-manifest: requiredStatement --> label/en/"Provider" --> value/en
- Rights: IIIF-manifest: "Rights"

##Environment
This notebook was created with the help of ChatGPT-5.2 and is supposed to be used in a Google Colab/Google Drive environment.

## 1) Installations, Imports and Path Configurations
 - Installation of needed libraries
 - Imports from libraries
 - Path-settings
 - Google Drive mount

In [ ]:
# === Installations ===
!pip -q install pandas requests tqdm

# === Imports & Setup ===
import time, json, re
from pathlib import Path
import pandas as pd
import requests
from tqdm import tqdm
from PIL import Image

# ========== DRIVE / PFAD-CONFIG ==========
# these paths need to be changed according to your directory structure
USE_GOOGLE_DRIVE = True
DATA_OUTPUT_DIR = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE"
CSV_PATH = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/data_gollner.csv"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

#Creation of the base-directory and the subordinated metadata- and log-directory
OUT_BASE   = Path(DATA_OUTPUT_DIR)
OUT_PAGES  = OUT_BASE / "pages_jpg"     # JPGs
OUT_META   = OUT_BASE / "metadata"      # CSV
OUT_LOGS   = OUT_BASE / "logs"
for p in (OUT_PAGES, OUT_META, OUT_LOGS):
    p.mkdir(parents=True, exist_ok=True)

STATE_FILE = OUT_LOGS / "state.json"  # creates a log for a next_start, if the notebook cannot download eveything at once
MASTER_LOG = OUT_LOGS / "download_log_master.csv"
PAGES_CSV  = OUT_META / "pages-jpg_metadata.csv"

# ========== BATCH-CONFIG ==========
BATCH_SIZE = 200
RESET_STATE = False
MANUAL_START_IDX = None
REWIND_BATCHES = 0

# ========== TESTING CONFIG ==========
MAX_TOTAL_PAGES = None   # set to None for full run

# ========== DOWNLOAD-CONFIG ==========
SIZE_PARAM = "max"     # full resolution of available jpgs - change if size is too great
MAX_SLEEP = 0.1        # break to let the system rest (courtesy)

# ========== CSV-columns (of provided csv by the ONB, change if needed) ==========
MANIFEST_COL = "iiif_manifest"
ANNO_ID_COL  = "anno_id"
ANNO_URL_COL = "anno_url"

# ========== FIXED METADATA ==========
# column values for metadata-file which should remain throughout:
FIXED_PERIODICAL_TITLE = "Die Bombe"

# ========== HTTP-Session ==========
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Colab-IIIF-JPG-Downloader/2.4"})


Mounted at /content/drive


Output of the cell above will be the following directory structure:
1) pages_jpg: containing the jpg-files of each page in a sub-directory of each issue
2) metadata: containing the metadata-file (where one row describes one page)
3) logs: where log-files are stored for resumable downloading

## 2) Auxiliary functions

In [ ]:
def get_json(url, tries=3, timeout=60):
    last = None
    for i in range(tries):
        try:
            r = SESSION.get(url, timeout=timeout)
            if r.ok:
                return r.json()
            last = f"{r.status_code} {r.text[:200]}"
        except Exception as e:
            last = str(e)
        time.sleep(0.6 * (i + 1))
    raise RuntimeError(f"GET JSON failed: {url} :: {last}")

def pick_lang_value(lang_map, preferred=("en", "none", "de")):
    if not isinstance(lang_map, dict):
        return None
    for lang in preferred:
        arr = lang_map.get(lang)
        if isinstance(arr, list) and arr:
            return arr[0]
    for _, arr in lang_map.items():
        if isinstance(arr, list) and arr:
            return arr[0]
    return None

def get_manifest_metadata_en(manifest: dict, wanted_label_en: str):
    wanted = wanted_label_en.strip().lower()
    for md in manifest.get("metadata", []) or []:
        lab_en = pick_lang_value(md.get("label") or {}, preferred=("en",))
        if (lab_en or "").strip().lower() == wanted:
            return pick_lang_value(md.get("value") or {}, preferred=("en", "none", "de"))
    return None

def get_required_statement_value_en(manifest: dict, wanted_label_en: str):
    rs = manifest.get("requiredStatement")
    if not isinstance(rs, dict):
        return None
    lab_en = pick_lang_value(rs.get("label") or {}, preferred=("en",))
    if (lab_en or "").strip().lower() == wanted_label_en.strip().lower():
        return pick_lang_value(rs.get("value") or {}, preferred=("en", "none", "de"))
    return None

def get_provider_label_en(manifest: dict):
    prov = (manifest.get("provider") or [])
    if not prov:
        return None
    return pick_lang_value((prov[0].get("label") or {}), preferred=("en", "none", "de"))

def iter_canvases(m: dict):
    if "items" in m:
        for c in m["items"]:
            if isinstance(c, dict) and c.get("type") == "Canvas":
                yield c
    for seq in m.get("sequences", []) or []:
        for c in seq.get("canvases", []) or []:
            yield c

def extract_image_service_and_body_url(canvas: dict):
    if "items" in canvas:
        try:
            body = canvas["items"][0]["items"][0]["body"]
            svc = body.get("service")
            if isinstance(svc, list) and svc:
                svc_id = svc[0].get("id") or svc[0].get("@id")
            elif isinstance(svc, dict):
                svc_id = svc.get("id") or svc.get("@id")
            else:
                svc_id = None
            return svc_id, (body.get("id") or body.get("@id"))
        except Exception:
            pass
    if "images" in canvas:
        try:
            res = canvas["images"][0]["resource"]
            svc = res.get("service") or {}
            return (svc.get("@id") or svc.get("id")), (res.get("@id") or res.get("id"))
        except Exception:
            pass
    return None, None

def build_jpg_url(service_id: str, size_param: str) -> str:
    return f"{service_id}/full/{size_param}/0/default.jpg"

def download_to(out_path, url, timeout=30):
    """
    Safe download:
    - Downloads to temporary *.tmp file
    - Only renames to final file if complete
    - Prevents truncated JPGs after crashes
    """
    out_path = Path(out_path)
    tmp_path = out_path.with_suffix(out_path.suffix + ".tmp")

    try:
        with requests.get(url, stream=True, timeout=timeout) as r:
            r.raise_for_status()
            expected_size = int(r.headers.get("Content-Length", 0))

            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        # Verify size if Content-Length available
        actual_size = tmp_path.stat().st_size
        if expected_size and actual_size != expected_size:
            tmp_path.unlink(missing_ok=True)
            return False

        # Replace existing file safely
        tmp_path.replace(out_path)
        return True

    except Exception:
        tmp_path.unlink(missing_ok=True)
        return False

def append_csv(path, rows, columns=None, encoding="utf-8"):
    """
    Append rows (list[dict]) to a CSV in a stable, reproducible column order.

    - If `columns` is provided: enforce exactly that order (missing cols become empty).
    - If `columns` is None:
        - If file exists: reuse its existing header order.
        - If file doesn't exist: infer order from first row's key order.
    - Automatically adds any "extra" keys (not in columns/header) at the end.
    """
    path = Path(path)
    if not rows:
        return

    df = pd.DataFrame(rows)

    # Determine target column order
    if columns is not None:
        col_order = list(columns)
    else:
        if path.exists() and path.stat().st_size > 0:
            # Read header only, preserve existing column order
            existing = pd.read_csv(path, nrows=0, encoding=encoding)
            col_order = list(existing.columns)
        else:
            # New file: respect insertion order of keys in the first row
            col_order = list(rows[0].keys())

    # Append any new/unseen columns at the end (so nothing gets dropped)
    for c in df.columns:
        if c not in col_order:
            col_order.append(c)

    # Reindex for stable order; missing columns become NaN/empty
    df = df.reindex(columns=col_order)

    # Write (append) with correct header handling
    write_header = not (path.exists() and path.stat().st_size > 0)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, mode="a", index=False, header=write_header, encoding=encoding)

def save_state(next_start: int):
    STATE_FILE.write_text(json.dumps({"next_start": int(next_start)}))

def parse_place_and_gnd(place_html: str | None):
    """
    Input example (already unescaped by JSON):
      <span>Wien -- (GND: <a href="http://d-nb.info/gnd/4066009-6">4066009-6</a>)</span>
    Output:
      ("Wien", "http://d-nb.info/gnd/4066009-6")  # or (place, None) if no GND link
    """
    if not place_html:
        return None, None

    txt = str(place_html)

    # Place: take text right after <span> up to " --" (or closing tag as fallback)
    m_place = re.search(r"<span>\s*([^<]+?)(?:\s*--|\s*</span>)", txt, flags=re.IGNORECASE)
    place = m_place.group(1).strip() if m_place else None

    # GND link: take href
    m_href = re.search(r'href="([^"]*d-nb\.info/gnd/[^"]+)"', txt, flags=re.IGNORECASE)
    gnd_link = m_href.group(1).strip() if m_href else None

    return place, gnd_link

## 3) Loading the CSV and starting the download

In [ ]:
# ========== loading provided CSV ==========
df = pd.read_csv(CSV_PATH)
for col in (MANIFEST_COL, ANNO_ID_COL, ANNO_URL_COL):
    if col not in df.columns:
        raise ValueError(f"Column '{col}' missing in CSV. Available: {list(df.columns)}")

N = len(df)
print(f"Datasets (issues): {N} | Manifest: {MANIFEST_COL} | Anno-ID: {ANNO_ID_COL} | Anno-URL: {ANNO_URL_COL}")

# ========== Stable column order for the PAGES metadata CSV ==========
# If the CSV already exists, append_csv() will keep that header order.
# If it does not exist yet, we enforce this order on the first write.
PAGES_COLUMNS = [
    # page-level (first)
    "page_id",
    "page_number",
    "iiif_jpg_url",
    "local_jpg_path",
    "width",
    "height",
    "download_status",
    # issue-level (repeated per page)
    "anno_id",
    "anno_url",
    "manifest_url",
    "periodical_title",
    "date",
    "publishing_place",
    "publishing_place_gnd",
    "language",
    "mediatype",
    "attribution",
    "provider",
    "rights",
]

# ========== Determine starting position (state + knobs) ==========
start_idx = 0
if STATE_FILE.exists():
    try:
        start_idx = json.loads(STATE_FILE.read_text()).get("next_start", 0)
    except Exception:
        start_idx = 0

# Apply knobs ONCE (so we don't reset/rewind every loop iteration)
if RESET_STATE:
    start_idx = 0
if MANUAL_START_IDX is not None:
    start_idx = max(0, min(N, int(MANUAL_START_IDX)))
if REWIND_BATCHES > 0:
    start_idx = max(0, start_idx - REWIND_BATCHES * BATCH_SIZE)

# Persist the chosen start_idx so the loop below continues from there
save_state(start_idx)

print(f"Starting at row index: {start_idx} (BATCH_SIZE={BATCH_SIZE})")

# ========== Running ALL Batches (CRASH-SAFE) ==========
# Writes metadata + logs + state AFTER EACH ISSUE and metadata AFTER EACH PAGE.
# If Colab disconnects, re-running this cell resumes from STATE_FILE['next_start'].

total_pages_downloaded = 0  # for MAX_TOTAL_PAGES testing limit (per run)

while True:
    # Load current position from state (authoritative for resume)
    try:
        cur = json.loads(STATE_FILE.read_text()).get("next_start", 0) if STATE_FILE.exists() else 0
    except Exception:
        cur = 0

    if cur >= N:
        print("\n✅ All issues processed (cur >= N).")
        break

    end_idx = min(cur + BATCH_SIZE, N)
    print(f"\n========== BATCH ==========")
    print(f"Processing issues rows {cur}..{end_idx-1} of {N-1} (BATCH_SIZE={BATCH_SIZE})")

    df_batch = df.iloc[cur:end_idx].copy()

    for row_pos_in_batch, (row_idx, row) in enumerate(
        tqdm(df_batch.iterrows(), total=len(df_batch), desc="Output (Batch)"),
        start=0
    ):
        # IMPORTANT: resume should be position-based, not index-based
        global_pos = cur + row_pos_in_batch  # absolute positional index in full df

        manifest_url = row.get(MANIFEST_COL, "")
        anno_id = str(row.get(ANNO_ID_COL, "")).strip()
        anno_url = str(row.get(ANNO_URL_COL, "")).strip()

        issue_log = {"row": int(global_pos), "anno_id": anno_id}

        if pd.isna(manifest_url) or not str(manifest_url).strip():
            issue_log.update({"pages_downloaded": 0, "status": "skip-empty-manifest"})
            append_csv(MASTER_LOG, [issue_log])
            save_state(global_pos + 1)
            continue

        manifest_url = str(manifest_url).strip()

        # loading the manifest
        try:
            m = get_json(manifest_url)
        except Exception as e:
            issue_log.update({"pages_downloaded": 0, "status": f"manifest-fail: {e}"})
            append_csv(MASTER_LOG, [issue_log])
            save_state(global_pos + 1)
            continue

        # Place parsing: keep only "Wien" and GND link in separate columns
        place_raw = get_manifest_metadata_en(m, "Place")
        publishing_place, publishing_place_gnd = parse_place_and_gnd(place_raw)

        # Issue-level metadata (repeated per page)
        issue_meta = {
            "anno_id": anno_id,
            "anno_url": anno_url,
            "manifest_url": manifest_url,
            # Title should not include date: fixed value for this corpus
            "periodical_title": FIXED_PERIODICAL_TITLE,
            "date": get_manifest_metadata_en(m, "Year/Date"),
            "publishing_place": publishing_place,
            "publishing_place_gnd": publishing_place_gnd,
            "language": get_manifest_metadata_en(m, "Languages"),
            "mediatype": get_manifest_metadata_en(m, "Mediatype"),
            "attribution": get_required_statement_value_en(m, "Attribution"),
            "provider": get_provider_label_en(m),
            "rights": m.get("rights"),
        }

        # Downloading the pages and metadata append per download
        pages_downloaded = 0
        issue_dir = OUT_PAGES / anno_id
        issue_dir.mkdir(parents=True, exist_ok=True)

        for page_idx, canv in enumerate(iter_canvases(m), start=1):
            # --- Global page limit for testing ---
            if MAX_TOTAL_PAGES is not None and total_pages_downloaded >= MAX_TOTAL_PAGES:
                break

            page_id = f"{anno_id}_{page_idx:03d}"  # e.g. bom18710108_001

            svc_id, body_id = extract_image_service_and_body_url(canv)

            jpg_url = None
            if svc_id:
                jpg_url = build_jpg_url(svc_id, SIZE_PARAM)
            elif body_id and str(body_id).lower().endswith((".jpg", ".jpeg")):
                jpg_url = body_id

            if not jpg_url:
                page_row = {
                    **issue_meta,
                    "page_id": page_id,
                    "page_number": page_idx,
                    "iiif_jpg_url": None,
                    "local_jpg_path": None,
                    "width": canv.get("width"),
                    "height": canv.get("height"),
                    "download_status": "no-image-url",
                }
                append_csv(PAGES_CSV, [page_row], columns=PAGES_COLUMNS)
                continue

            out_path = issue_dir / f"{page_id}.jpg"
            ok = download_to(out_path, jpg_url)

            page_row = {
                **issue_meta,
                "page_id": page_id,
                "page_number": page_idx,
                "iiif_jpg_url": jpg_url,
                "local_jpg_path": str(out_path) if ok else None,
                "width": canv.get("width"),
                "height": canv.get("height"),
                "download_status": "ok" if ok else "download-fail",
            }
            append_csv(PAGES_CSV, [page_row], columns=PAGES_COLUMNS)

            if ok:
                pages_downloaded += 1
                total_pages_downloaded += 1

            if MAX_SLEEP:
                time.sleep(MAX_SLEEP)

        # Issue log row + state written immediately (crash-safe)
        issue_log.update({
            "pages_downloaded": pages_downloaded,
            "status": "ok" if pages_downloaded > 0 else "no-pages",
        })
        append_csv(MASTER_LOG, [issue_log])

        # Always advance (so a problematic issue doesn't block progress)
        save_state(global_pos + 1)

        # --- Stop after reaching global page limit ---
        if MAX_TOTAL_PAGES is not None and total_pages_downloaded >= MAX_TOTAL_PAGES:
            print(f"\nReached MAX_TOTAL_PAGES = {MAX_TOTAL_PAGES}. Stopping test run.")
            break

    # If we stopped due to MAX_TOTAL_PAGES, stop the outer loop too
    if MAX_TOTAL_PAGES is not None and total_pages_downloaded >= MAX_TOTAL_PAGES:
        break

print("\nFinished download loop.")
print(f"Pages (JPG): {OUT_PAGES}")
print(f"Pages metadata CSV: {PAGES_CSV}")
print(f"Master-Log: {MASTER_LOG}")
print(f"State: {STATE_FILE}")
if MAX_TOTAL_PAGES is not None:
    print(f"Total pages downloaded this run: {total_pages_downloaded} (limit: {MAX_TOTAL_PAGES})")


Datasets (issues): 2593 | Manifest: iiif_manifest | Anno-ID: anno_id | Anno-URL: anno_url
Starting at row index: 2593 (BATCH_SIZE=200)

✅ All issues processed (cur >= N).

Finished download loop.
Pages (JPG): /content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/pages_jpg
Pages metadata CSV: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata.csv
Master-Log: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/logs/download_log_master.csv
State: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/logs/state.json


##4) (Optional) Checking the download-count and retrying failed downloads

In [ ]:
# =========================
# FINAL INTEGRITY CHECK: How many pages were downloaded?
# =========================

# Count actual downloaded image files on disk
img_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
all_imgs = [p for p in OUT_PAGES.rglob("*") if p.is_file() and p.suffix.lower() in img_exts]

issue_dirs = sorted([p for p in OUT_PAGES.iterdir() if p.is_dir()])

print("========== DOWNLOAD INTEGRITY CHECK ==========")
print(f"Pages folder: {OUT_PAGES}")
print(f"Issue folders found: {len(issue_dirs)}")
print(f"Downloaded image files found: {len(all_imgs)}")

# Cross-check with metadata CSV
if PAGES_CSV.exists() and PAGES_CSV.stat().st_size > 0:
    import pandas as pd
    df_pages = pd.read_csv(PAGES_CSV, dtype=str).fillna("")
    total_rows = len(df_pages)
    ok = (df_pages.get("download_status", "") == "ok").sum() if "download_status" in df_pages.columns else None
    print("\n========== METADATA CROSS-CHECK ==========")
    print(f"Rows in pages metadata CSV: {total_rows}")
    if ok is not None:
        print(f"Rows with download_status == 'ok': {ok}")
        print(f"Rows with non-ok status: {total_rows - ok}")
else:
    print("\n(No pages metadata CSV found yet; run the download cell first.)")


========== DOWNLOAD INTEGRITY CHECK ==========
Pages folder: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/pages_jpg
Issue folders found: 2564
Downloaded image files found: 21124

========== METADATA CROSS-CHECK ==========
Rows in pages metadata CSV: 21126
Rows with download_status == 'ok': 21126
Rows with non-ok status: 0


In [ ]:
# =========================
# BUILDING RETRY LIST (status != ok OR file missing)
# =========================

PAGES_DIR = Path("/content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/pages_jpg")
CSV_PATH  = Path("/content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata.csv")

df = pd.read_csv(CSV_PATH).fillna("")

for c in ["page_id", "anno_id", "iiif_jpg_url", "download_status"]:
    assert c in df.columns, f"Missing '{c}' in CSV. Columns: {list(df.columns)}"

if "local_jpg_path" not in df.columns:
    df["local_jpg_path"] = ""

def target_path(row) -> Path:
    lp = str(row.get("local_jpg_path", "")).strip()
    if lp and lp.lower() != "nan":
        return Path(lp)
    return PAGES_DIR / str(row["anno_id"]).strip() / f"{str(row['page_id']).strip()}.jpg"

retry_rows = []
non_ok = 0
missing = 0
no_url = 0

for _, row in df.iterrows():
    url = str(row["iiif_jpg_url"]).strip()
    if not url:
        no_url += 1
        continue

    status = str(row["download_status"]).strip().lower()
    out_path = target_path(row)

    is_non_ok = (status != "ok")
    is_missing = (not out_path.exists())

    if is_non_ok: non_ok += 1
    if is_missing: missing += 1

    if is_non_ok or is_missing:
        retry_rows.append({
            "page_id": row["page_id"],
            "anno_id": row["anno_id"],
            "iiif_jpg_url": url,
            "download_status": row["download_status"],
            "local_jpg_path": str(out_path),
            "reason": ("non_ok" if is_non_ok else "") + ("|missing" if is_missing else "")
        })

retry_df = pd.DataFrame(retry_rows)

print("Rows in metadata:", len(df))
print("Retry candidates:", len(retry_df))
print("Counters: non_ok =", non_ok, "| missing =", missing, "| no_url =", no_url)

RETRY_LIST_PATH = CSV_PATH.with_name(CSV_PATH.stem + "_retry_list.csv")
retry_df.to_csv(RETRY_LIST_PATH, index=False)
print("Saved retry list:", RETRY_LIST_PATH)

retry_df.head(20)

Rows in metadata: 21126
Retry candidates: 235
Counters: non_ok = 235 | missing = 233 | no_url = 0
Saved retry list: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata_retry_list.csv


,page_id,anno_id,iiif_jpg_url,download_status,local_jpg_path,reason
0,bom18730323_003,bom18730323,https://api.onb.ac.at/iiif/image/v3/11D491E9/u...,download-fail,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...,non_ok|missing
1,bom18730323_004,bom18730323,https://api.onb.ac.at/iiif/image/v3/11D491E9/u...,download-fail,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...,non_ok|missing
2,bom18730323_005,bom18730323,https://api.onb.ac.at/iiif/image/v3/11D491E9/u...,download-fail,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...,non_ok|missing
3,bom18730323_006,bom18730323,https://api.onb.ac.at/iiif/image/v3/11D491E9/u...,download-fail,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...,non_ok|missing
4,bom18730323_007,bom18730323,https://api.onb.ac.at/iiif/image/v3/11D491E9/u...,download-fail,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...,non_ok|missing
5,bom18730323_008,bom18730323,https://api.onb.ac.at/iiif/image/v3/11D491E9/u...,download-fail,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...,non_ok|missing
6,bom18730323_009,bom18730323,https://api.onb.ac.at/iiif/image/v3/11D491E9/u...,download-fail,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...,non_ok|missing
7,bom18730323_010,bom18730323,https://api.onb.ac.at/iiif/image/v3/11D491E9/u...,download-fail,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...,non_ok|missing
8,bom18730323_012,bom18730323,https://api.onb.ac.at/iiif/image/v3/11D491E9/u...,download-fail,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...,non_ok|missing
9,bom18730501_017,bom18730501,https://api.onb.ac.at/iiif/image/v3/11C4610C/u...,download-fail,/content/drive/MyDrive/Masterarbeit_DH/Pipelin...,non_ok|missing


In [ ]:
# =========================
# DOWNLOAD FROM RETRY LIST ONLY
# =========================

PAGES_DIR = Path("/content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/pages_jpg")
CSV_PATH  = Path("/content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata.csv")
RETRY_LIST_PATH = CSV_PATH.with_name(CSV_PATH.stem + "_retry_list.csv")

retry_df = pd.read_csv(RETRY_LIST_PATH).fillna("")
df = pd.read_csv(CSV_PATH).fillna("")

# Ensure columns
if "download_error" not in df.columns:
    df["download_error"] = ""
if "local_jpg_path" not in df.columns:
    df["local_jpg_path"] = ""

def is_readable_image(path: Path) -> bool:
    if not path.exists() or path.stat().st_size < 1024:
        return False
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False

def download_to_path(url: str, out_path: Path, timeout=(10, 60)) -> tuple[bool, str]:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_suffix(out_path.suffix + ".part")
    try:
        r = requests.get(url, stream=True, timeout=timeout)
        if r.status_code != 200:
            return False, f"http_{r.status_code}"
        with tmp.open("wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        tmp.replace(out_path)

        if not is_readable_image(out_path):
            out_path.unlink(missing_ok=True)
            return False, "corrupt_or_unreadable"
        return True, "ok"
    except Exception as e:
        try:
            tmp.unlink(missing_ok=True)
        except Exception:
            pass
        return False, f"exc_{type(e).__name__}"

# Index for fast updates
key_to_idx = {(str(r.get("page_id","")), str(r.get("anno_id",""))): i for i, r in df.iterrows()}

ok_count = 0
skip_count = 0
fail_count = 0

for n, rrow in enumerate(tqdm(retry_df.to_dict("records"), total=len(retry_df), desc="Retry downloads"), start=1):
    page_id = str(rrow["page_id"])
    anno_id = str(rrow["anno_id"])
    url = str(rrow["iiif_jpg_url"]).strip()
    out_path = Path(str(rrow["local_jpg_path"]))

    # Resumable: if it already exists + is readable, don't redownload
    if is_readable_image(out_path):
        skip_count += 1
        idx = key_to_idx.get((page_id, anno_id))
        if idx is not None:
            df.at[idx, "download_status"] = "ok"
            df.at[idx, "download_error"] = ""
            df.at[idx, "local_jpg_path"] = str(out_path)
        continue

    success, msg = download_to_path(url, out_path)
    idx = key_to_idx.get((page_id, anno_id))

    if idx is not None:
        df.at[idx, "local_jpg_path"] = str(out_path)

    if success:
        ok_count += 1
        if idx is not None:
            df.at[idx, "download_status"] = "ok"
            df.at[idx, "download_error"] = ""
    else:
        fail_count += 1
        if idx is not None:
            df.at[idx, "download_status"] = "retry_failed"
            df.at[idx, "download_error"] = msg

    if n % 10 == 0:
        print(f"Progress {n}/{len(retry_df)} | ok={ok_count} skip={skip_count} fail={fail_count}")

OUT_UPDATED = CSV_PATH.with_name(CSV_PATH.stem + "_UPDATED.csv")
df.to_csv(OUT_UPDATED, index=False)

print("\nDone.")
print("Downloaded OK:", ok_count)
print("Skipped (already present):", skip_count)
print("Still failing:", fail_count)
print("Updated metadata:", OUT_UPDATED)

Retry downloads:   4%|▍         | 10/235 [00:19<07:06,  1.90s/it]

Progress 10/235 | ok=10 skip=0 fail=0


Retry downloads:   9%|▊         | 20/235 [00:43<08:16,  2.31s/it]

Progress 20/235 | ok=20 skip=0 fail=0


Retry downloads:  13%|█▎        | 30/235 [01:04<07:46,  2.27s/it]

...

Progress 220/235 | ok=218 skip=2 fail=0


Retry downloads:  98%|█████████▊| 230/235 [08:26<00:09,  1.96s/it]

Progress 230/235 | ok=228 skip=2 fail=0


Retry downloads: 100%|██████████| 235/235 [08:34<00:00,  2.19s/it]



Done.
Downloaded OK: 233
Skipped (already present): 2
Still failing: 0
Updated metadata: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_BC/Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata_UPDATED.csv
